# Mental Health Signal Detector — Fine-tuning v3
**Artefact School of Data — Bootcamp Data Science, Mars 2026**

## Stratégie v3 — Domaine clinique strict

### Problème identifié dans v2
Le modèle v2 est entraîné sur un mix de données générales (DAIR-AI, GoEmotions) et cliniques (eRisk25, Kaggle). Les données générales introduisent du bruit : un commentaire Reddit exprimant de la "peur" face à un film d'horreur est étiqueté `détresse=1`, ce qui biaise la représentation apprise.

### Deux changements principaux

**1. Données cliniques uniquement**
- ✅ **eRisk25** : labels cliniques dépression validés (CLEF 2025, 909 sujets Reddit)
- ✅ **Kaggle Reddit Depression** : communautés r/depression, r/SuicideWatch
- ❌ DAIR-AI/emotion : phrases générales ("I feel great"), non cliniques
- ❌ GoEmotions : commentaires Google généraux, 28 émotions, trop bruités

**2. Mental-BERT comme base**
- `mental/mental-bert-base-uncased` : pré-entraîné sur 1.3B tokens santé mentale
  (Reddit r/depression, r/anxiety, r/SuicideWatch + PubMed psychiatrie)
- Même architecture BERT-base → même pipeline d'entraînement
- Benchmarks publiés : +3-5% F1 sur tâches cliniques vs DistilBERT standard

### Métrique primaire : Sensitivité (recall classe 1)
Dans un outil de santé mentale, **manquer un cas de détresse est plus grave qu'un faux positif**.
On cible :
- Sensitivité ≥ 0.90 (recall détresse)
- AUC-ROC ≥ 0.92
- F1 Macro ≥ 0.85

In [ ]:
# Étape 1 : installation
# ⚠️ Redémarrer la session après cette cellule
!pip install -q -U transformers accelerate datasets scikit-learn

In [ ]:
# Étape 2 : vérification GPU
import torch
print('GPU disponible :', torch.cuda.is_available())
print('Device :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ⚠️')
import transformers
print('Transformers :', transformers.__version__)

In [ ]:
# Étape 3 : Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os

DRIVE_DIR  = '/content/drive/MyDrive/mh_detector'
OUTPUT_DIR = f'{DRIVE_DIR}/mental_bert_v3'
DATA_DIR   = f'{DRIVE_DIR}/data'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Modèle    :', OUTPUT_DIR)
print('Données   :', DATA_DIR)
print('Fichiers  :', os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else '⚠️ Dossier data absent')

In [ ]:
# Étape 4 : chargement des données (cliniques uniquement)
import re
import json
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r"'", '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()


def load_kaggle(path: str, max_samples: int = 100_000) -> pd.DataFrame:
    """Kaggle Reddit Depression — labels communautaires (r/depression, r/SuicideWatch)."""
    df = pd.read_csv(path, low_memory=False)
    df['title'] = df['title'].fillna('')
    df['body']  = df['body'].fillna('')
    df['text']  = (df['title'] + ' ' + df['body']).str.strip()
    df = df[['text', 'label']].dropna()
    df['label'] = df['label'].astype(float).astype(int)
    df = df[df['text'].str.len() > 20]

    # Rééquilibrage 40/60 (détresse/non-détresse)
    d1 = df[df['label'] == 1]
    d0 = df[df['label'] == 0].sample(n=min(int(len(d1) * 1.5), len(df[df['label']==0])), random_state=42)
    df = pd.concat([d1, d0], ignore_index=True)

    if max_samples and len(df) > max_samples:
        df, _ = train_test_split(df, train_size=max_samples, stratify=df['label'], random_state=42)

    df['text'] = df['text'].apply(clean_text)
    df = df[df['text'].str.len() > 10]
    print(f'Kaggle : {len(df)} lignes | {df["label"].value_counts().to_dict()}')
    return df.reset_index(drop=True)


def load_erisk25(data_path: str) -> pd.DataFrame:
    """eRisk25 — labels cliniques dépression validés (CLEF 2025)."""
    json_dir = f'{data_path}/final-eriskt2-dataset-with-ground-truth/all_combined'
    files = glob.glob(f'{json_dir}/subject_*.json')
    if not files:
        raise FileNotFoundError(f'Aucun fichier subject_*.json dans {json_dir}')

    rows = []
    for filepath in files:
        with open(filepath, encoding='utf-8') as f:
            try:
                posts = json.load(f)
            except json.JSONDecodeError:
                continue
        for post in posts:
            sub = post.get('submission', {})
            target = sub.get('target')
            if target is None:
                continue
            text = ((sub.get('title') or '') + ' ' + (sub.get('body') or '')).strip()
            if len(text) > 20:
                rows.append({'text': text, 'label': int(bool(target))})

    df = pd.DataFrame(rows)
    df['text'] = df['text'].apply(clean_text)
    df = df[df['text'].str.len() > 10].drop_duplicates(subset=['text'])
    print(f'eRisk25 : {len(df)} lignes | {df["label"].value_counts().to_dict()}')
    return df.reset_index(drop=True)


# ─── Assembler les sources cliniques ────────────────────────────────────────
KAGGLE_PATH   = f'{DATA_DIR}/reddit_depression_dataset.csv'
ERISK25_PATH  = f'{DATA_DIR}/erisk25'

frames = []
if os.path.exists(KAGGLE_PATH):
    frames.append(load_kaggle(KAGGLE_PATH, max_samples=100_000))
else:
    print('⚠️  Kaggle non trouvé — entraînement sur eRisk25 uniquement')

if os.path.exists(ERISK25_PATH):
    frames.append(load_erisk25(ERISK25_PATH))
else:
    print('⚠️  eRisk25 non trouvé — entraînement sur Kaggle uniquement')

if not frames:
    raise RuntimeError('Aucune source de données trouvée. Uploader les fichiers dans Drive/mh_detector/data/')

df = pd.concat(frames, ignore_index=True).drop_duplicates(subset=['text'])
print(f'\nDataset clinique final : {len(df)} lignes')
print(f'Distribution :\n{df["label"].value_counts()}')

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
print(f'Train: {len(train_df)} | Test: {len(test_df)}')

In [ ]:
# Étape 5 : tokenisation avec Mental-BERT
#
# mental/mental-bert-base-uncased :
#   - Pré-entraîné sur 1.3B tokens santé mentale
#     (Reddit : r/depression, r/anxiety, r/SuicideWatch + articles PubMed psychiatrie)
#   - Architecture identique à bert-base-uncased → même pipeline
#   - Publié : Ji et al. 2022 "MentalBERT: Publicly Available Pretrained Language Models
#     for Mental Healthcare" — NAACL 2022
#
# Alternative si quota Colab dépassé : 'distilbert-base-uncased' (v2)

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset

MODEL_NAME = 'mental/mental-bert-base-uncased'
# MODEL_NAME = 'distilbert-base-uncased'  # fallback

print(f'Chargement tokenizer : {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

train_ds = Dataset.from_pandas(train_df[['text', 'label']]).map(tokenize, batched=True)
test_ds  = Dataset.from_pandas(test_df[['text', 'label']]).map(tokenize, batched=True)

for ds in [train_ds, test_ds]:
    ds = ds.rename_column('label', 'labels')
    ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

train_ds = train_ds.rename_column('label', 'labels')
test_ds  = test_ds.rename_column('label', 'labels')
train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print('Tokenisation OK')
print('Exemple token ids :', train_ds[0]['input_ids'][:10])

In [ ]:
# Étape 6 : entraînement
#
# Changements v3 vs v2 :
#   - metric_for_best_model = 'recall_1' (sensitivité) au lieu de 'f1'
#     → Priorité à ne pas manquer les cas de détresse
#   - weight_decay = 0.01 → régularisation L2 (réduit overfitting v2.1)
#   - warmup_ratio = 0.1 → learning rate progressif (réduit divergence)
#   - EarlyStopping patience=2 (arrêt si recall_1 n'améliore pas 2 epochs de suite)

from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    proba_pos = logits[:, 1]
    try:
        auc = roc_auc_score(labels, proba_pos)
    except Exception:
        auc = 0.0
    return {
        'accuracy':    accuracy_score(labels, preds),
        'f1_macro':    f1_score(labels, preds, average='macro'),
        'recall_1':    recall_score(labels, preds, pos_label=1),   # ← sensitivité
        'recall_0':    recall_score(labels, preds, pos_label=0),   # ← spécificité
        'auc_roc':     auc,
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,                        # 10% du training = warm-up progressif
    weight_decay=0.01,                       # L2 régularisation
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='recall_1',        # Sensitivité = priorité clinique
    greater_is_better=True,
    logging_steps=100,
    report_to='none',
    fp16=True,                               # Mixed precision T4
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f'Début entraînement : {MODEL_NAME}')
print(f'Train : {len(train_ds)} | Test : {len(test_ds)}')
trainer.train()

In [ ]:
# Étape 7 : évaluation clinique complète
from sklearn.metrics import classification_report, brier_score_loss
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

predictions = trainer.predict(test_ds)
logits      = predictions.predictions
labels      = predictions.label_ids
preds       = np.argmax(logits, axis=-1)
proba_pos   = logits[:, 1]
# Normalisation logits → probabilités avec softmax
proba_pos_norm = np.exp(proba_pos) / (np.exp(logits[:, 0]) + np.exp(proba_pos))

sensitivity  = recall_score(labels, preds, pos_label=1)
specificity  = recall_score(labels, preds, pos_label=0)
auc          = roc_auc_score(labels, proba_pos_norm)
brier        = brier_score_loss(labels, proba_pos_norm)
f1_macro     = f1_score(labels, preds, average='macro')
acc          = accuracy_score(labels, preds)

print('=' * 60)
print(f'  RÉSULTATS v3 — {MODEL_NAME}')
print('  ' + '-' * 56)
print(f'  Accuracy       : {acc:.4f}')
print(f'  F1 Macro       : {f1_macro:.4f}')
print(f'  Sensitivité    : {sensitivity:.4f}  ← objectif ≥ 0.90')
print(f'  Spécificité    : {specificity:.4f}')
print(f'  AUC-ROC        : {auc:.4f}          ← objectif ≥ 0.92')
print(f'  Brier score    : {brier:.4f}          ← calibration (0=parfait)')
print('=' * 60)
print()
print(classification_report(labels, preds, target_names=['non-détresse', 'détresse']))

# Courbe de calibration
prob_true, prob_pred = calibration_curve(labels, proba_pos_norm, n_bins=10)
plt.figure(figsize=(6, 5))
plt.plot([0, 1], [0, 1], 'k--', label='Calibration parfaite')
plt.plot(prob_pred, prob_true, 's-', label=MODEL_NAME)
plt.xlabel('Score prédit'); plt.ylabel('Fraction vrais positifs')
plt.title('Calibration des probabilités')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Étape 8 : sauvegarde
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Modèle sauvegardé → {OUTPUT_DIR}')
print('Fichiers :', os.listdir(OUTPUT_DIR))

# Résumé des métriques dans un fichier
import json
summary = {
    'model_name':  MODEL_NAME,
    'accuracy':    round(float(acc), 4),
    'f1_macro':    round(float(f1_macro), 4),
    'recall_1':    round(float(sensitivity), 4),
    'recall_0':    round(float(specificity), 4),
    'auc_roc':     round(float(auc), 4),
    'brier_score': round(float(brier), 4),
    'dataset_size': len(df),
    'sources': 'eRisk25 + Kaggle Reddit Depression (clinical only)',
}
with open(f'{OUTPUT_DIR}/eval_metrics.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

## Objectifs à atteindre

| Métrique | v2 DistilBERT | Cible v3 Mental-BERT |
|----------|:-------------:|:--------------------:|
| Accuracy | 89.0% | ≥ 89% |
| F1 Macro | 86.5% | ≥ 87% |
| **Sensitivité** | ~85% | **≥ 90%** |
| AUC-ROC | N/A | ≥ 0.92 |
| Brier score | N/A | ≤ 0.10 |

## Comment utiliser ce modèle en production

1. Télécharger le dossier `mental_bert_v3/` depuis Google Drive
2. Le placer dans `models/fine_tuned/` du projet
3. L'API utilise déjà `--model-name` comme paramètre configurable
4. Variable d'environnement : `VITE_MODEL_TYPE=distilbert` dans Vercel

## Phase suivante — Multi-label clinique (v4)

L'étape suivante serait de prédire directement les dimensions cliniques `[depression, anxiety]`
depuis le texte (classification multi-label), pour remplacer la détection heuristique par mots-clés
actuellement dans le navigateur.

Données nécessaires :
- Depression : eRisk25 ✅
- Anxiety : CLEF eRisk Task 1 (anxiété) ou Kaggle anxiety dataset
- Burnout : dataset spécifique manquant (SMHD si accès obtenu)